In [ ]:
from option_analyzer import *
from indicators import compute_emas
self = OptionAnalyzer('quotes', 'chain')
pd.set_option("display.max_columns", None)

etf_symbols = 'QQQ|IWM|SPY|GLD|DIA|USO'
mag_symbols = 'NVDA|GOOGL|MSFT|AAPL|META|AMZN|TSLA|TSM'
filter_etf = lambda _df: _df[_df.symbol.str.contains(etf_symbols)]
filter_mag = lambda _df: _df[_df.symbol.str.contains(mag_symbols)]
filter_others = lambda _df: _df[~_df.symbol.str.contains(etf_symbols + '|' + mag_symbols)]

In [ ]:
def load_walls(wall_file):
    data_age = (time.time() - os.path.getmtime(wall_file))/60
    print(f'{wall_file} is {data_age:.1f} minutes old')
    walls = {}
    with open(wall_file) as fo:
        for line in fo:
            s, w = line.rstrip().split(' ')
            walls[s] = float(w)
    return walls

def profit_overview_of_short_puts(dfp, mn_lb=0.9, mn_ub=1.0, dte_ub=40, dth_profit_lb=24):
    mn2df = {}
    metrics = ['dthProfit', 'dte', 'dthStrikeMargin', 'Delta']
    for mn in np.arange(mn_lb, mn_ub, 0.01):
        key = f'{mn}'
        _df = dfp[(dfp.moneyness <= mn) & (dfp.dte <= dte_ub) & (dfp.dthProfit >= dth_profit_lb) & (dfp.dthStrikeMargin >= 0)].reset_index()
        mn2df[key] = _df.loc[_df.groupby('symbol')['dthProfit'].idxmax(), ['symbol'] + metrics].set_index('symbol')
    for metric in metrics:
        _df = pd.DataFrame(dict([(key, mn2df[key][metric]) for key in mn2df])).T
        _df.index.name = 'moneyness'
        title = f'DTE under {dte_ub} days' if metric == 'dte' else f'dthProfit >= {dth_profit_lb} percent' if metric == 'dthProfit' else metric
        #px.bar(_df, barmode='group', title=title, width=1500, height=420).show()
        px.line(_df, title=title, width=1500, height=420).show()

def show_put_options_by_dte_and_moneyness(dfp, dte, mn_lb=0.9, mn_ub=1.0, n_rows=20):
    _filter = (dfp.dte==dte) & (dfp.moneyness >= mn_lb) & (dfp.moneyness <= mn_ub) & (dfp.dthStrikeMargin >= 0)
    _dfp = dfp[_filter].copy()
    if _dfp.shape[0] == 0:
        print('Are you using a valid dpe?')
        return
    title = f'{dte} dte {_dfp.iloc[0].expDt}'
    _dfp = _dfp.sort_values(by='dthProfit', ascending=False).head(n_rows).sort_values(by='dthStrikeMargin', ascending=False)
    px.scatter(_dfp, x='dthStrikeMargin', y='dthProfit', color='symbol', title=title, width=1000, height=600).show()
    top_cols = ['symbol', 'expDt', 'strike', 'dte', 'dthStrikeMargin', 'dthProfit', 'mid', 'moneyness', 'Delta', 'OpenInterest', 'lastPrice', 'pctSpread']
    return _dfp.loc[:, top_cols + [c for c in _dfp.columns if c not in top_cols]]

### Run this once every day to load close prices in the past 60 days

In [ ]:
df_close = pd.read_csv('output/yf_close.csv', parse_dates=['Date']).set_index('Date').tail(60)
df_close.columns.name = 'symbol'
latest_bollinger_file = max(glob('output/bollinger*.csv'))
df_boll = pd.read_csv(latest_bollinger_file).set_index('symbol')
df_boll = df_boll.loc[:, ['LB20', 'UB20', 'MA20', 'LB30', 'UB30', 'MA30']]
today = pd.Timestamp.now().normalize()
print('df_close data age:', today - df_close.index[-1])
print('latest Bollinger file:', latest_bollinger_file)

### Run this cell to read from data directory

In [ ]:
os.system('sync > /dev/null 2>&1')
option_type = 'put'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv')) if 'macmini' not in f]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(['%40s ' % os.path.basename(_) + ' '.join(self.list_symbols_in_data_file(_)) for _ in latest_option_files]))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print('Last symbol:', latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))

# Read option data processed by servers
dfp = pd.concat([pd.read_csv(_f) for _f in latest_option_files])
put_walls = load_walls('put_walls.txt')
put_walls['GLW'] = 135
put_walls['MU'] = 850
print(''.join([f' {k}:{v}'.replace('.0', '') for k, v in put_walls.items()]))
dfp['put_wall'] = dfp.symbol.apply(lambda x: put_walls.get(x, np.nan))
dfp['option'] = dfp.symbol + ' ' + dfp.expDt + ' ' + dfp.strike.astype(str)

# Quick overview of the three major ETFs
dfp_etf = filter_etf(dfp)
dfp_etf =dfp_etf[(dfp_etf.strike <= dfp_etf.put_wall) & (dfp_etf.strike < dfp_etf.lastPrice) & (dfp_etf.dte <= 45) & (dfp_etf.dte >= 21)].sort_values(by='dteProfit', ascending=False)
px.scatter(dfp_etf.head(20), x='dthStrikeMargin', y='dteProfit', color='option', height=600, width=1500).show()
dfp_etf.drop(columns=['hdte_resid', 'resid', 'dth', 'dtz', 'option']).head(10)

In [ ]:
_symbol = 'SNDK'
_dfp = dfp[(dfp.symbol==_symbol) & (dfp.strike <= dfp.put_wall) & (dfp.dte <= 45)].sort_values(by='mid', ascending=False)
px.scatter(_dfp, x='dthStrikeMargin', y='mid', color='option', width=1500, height=800).show()
_dfp.head(20)

In [ ]:
print('put walls:', put_walls)
_f = ~dfp.symbol.str.contains(r'META|MRVL') & (dfp.strike < dfp.lastPrice) & (dfp.dthStrikeMargin > 0) & (dfp.strike <= 342)
__df = dfp[_f & (dfp.strike <= dfp.put_wall) & (dfp.dte <= 45) & (dfp.dte >= 7)].sort_values(by='dthProfit', ascending=False)
px.scatter(__df.head(20), x='dthStrikeMargin', y='dteProfit', color='option', width=1500, height=600).show()
__df.drop(columns=['hdte_resid', 'resid', 'dth', 'dtz', 'option']).head(20)

In [ ]:
dfp_mag = filter_mag(dfp)
__df = dfp_mag[(dfp_mag.strike < dfp_mag.put_wall) & (dfp_mag.dte <= 45) & (dfp_mag.dthStrikeMargin > 0)].sort_values(by='dteProfit', ascending=False)
px.scatter(__df.head(20), x='dthStrikeMargin', y='dteProfit', color='option', width=1500, height=600).show()
__df.drop(columns=['hdte_resid', 'resid', 'dth', 'dtz', 'option']).head(20)

In [ ]:
dfp_o = filter_others(dfp)
__df = dfp_o[(dfp_o.strike < dfp_o.put_wall) & (dfp_o.dte <= 45) & (dfp_o.dthStrikeMargin > 0)].sort_values(by='dteProfit', ascending=False)
px.scatter(__df.head(20), x='dthStrikeMargin', y='dteProfit', color='option', width=1500, height=600).show()
__df.drop(columns=['hdte_resid', 'resid', 'dth', 'dtz', 'option']).head(20)

### Overview of put options for three groups of symbols: ETFs, Mag 7 + TMC, and Others

In [ ]:
profit_overview_of_short_puts(filter_others(dfp), 0.72, 0.8, dte_ub=45, dth_profit_lb=50)
profit_overview_of_short_puts(filter_mag(dfp), dte_ub=45, dth_profit_lb=24)
profit_overview_of_short_puts(filter_etf(dfp), dte_ub=45, dth_profit_lb=24)

### ETF dthProfit vs dthStrikeMargin on specific DTE

In [ ]:
target_dte = 2
_dfp_etf = show_put_options_by_dte_and_moneyness(filter_etf(dfp), target_dte, mn_lb=0.9, mn_ub=0.96, n_rows=10)
_dfp_etf

### Mag 7 + TSM dthProfit vs dthStrikeMargin on specific DTE

In [ ]:
target_dte = 2
_dfp_mag = filter_mag(dfp)
_max_dth_profit = _dfp_mag[_dfp_mag.dthStrikeMargin >= 5].dthProfit.max()
print(_max_dth_profit)
_dfp_mag = _dfp_mag[_dfp_mag.dthProfit >= 0.5*_max_dth_profit]
_dfp_mag = show_put_options_by_dte_and_moneyness(_dfp_mag, target_dte, mn_lb=0.8, mn_ub=0.93, n_rows=20)
_dfp_mag

### All Other dthProfit vs dthStrikeMargin on specific DTE

In [ ]:
target_dte = 2
_dfp = show_put_options_by_dte_and_moneyness(filter_others(dfp), target_dte, mn_lb=0.8, mn_ub=0.82, n_rows=20)
_dfp